# FINd end-to-end sanity check

Goal: run FINd on a small subset (~30 images), confirm the pipeline works (decode → resize → transform → binarize), and visualize within-cluster vs between-cluster Hamming distances.

In [ ]:
import sys, os
from pathlib import Path

# Resolve project root: works whether notebook is at root or in notebooks/
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == 'notebooks' else _here
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # so any relative file paths work as if running from root

IMG_DIR = PROJECT_ROOT / 'meme_images'
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

import time
import random
from collections import defaultdict

from FINd import FINDHasher


## 1. Sample a subset

Filenames look like `CLUSTER_IMAGEID.jpg` (first 4 chars = cluster id). Pick a few clusters with ≥2 images each so we can measure within-cluster distances.

In [ ]:
random.seed(42)

IMG_DIR = "meme_images"
all_files = sorted(os.listdir(IMG_DIR))

clusters = defaultdict(list)
for f in all_files:
    clusters[f[:4]].append(f)

multi_clusters = [cid for cid, imgs in clusters.items() if len(imgs) >= 2]
print(f"Total files: {len(all_files)}")
print(f"Total clusters: {len(clusters)}")
print(f"Clusters with >=2 images: {len(multi_clusters)}")

picked_clusters = random.sample(multi_clusters, 10)
subset = []
for cid in picked_clusters:
    subset.extend(clusters[cid][:3])

print(f"Subset size: {len(subset)} images from {len(picked_clusters)} clusters")

## 2. Hash everything + measure time per image

In [ ]:
findHasher = FINDHasher()

hashes = {}
t0 = time.time()
for f in subset:
    hashes[f] = findHasher.fromFile(f"{IMG_DIR}/{f}")
elapsed = time.time() - t0

print(f"Hashed {len(hashes)} images in {elapsed:.1f}s ({elapsed / len(hashes):.2f}s per image)")
print(f"Sample hash (hex): {hashes[subset[0]]}")

## 3. Within-cluster vs between-cluster Hamming distances

Expectation:
- within-cluster: ~30–90 bits (same meme with modifications)
- between-cluster: ~128 bits (random baseline)

In [ ]:
within = []
between = []

for cid in picked_clusters:
    imgs = [f for f in clusters[cid] if f in hashes]
    for i in range(len(imgs)):
        for j in range(i + 1, len(imgs)):
            within.append((imgs[i], imgs[j], int(hashes[imgs[i]] - hashes[imgs[j]])))

reps = [clusters[cid][0] for cid in picked_clusters]
for i in range(len(reps)):
    for j in range(i + 1, len(reps)):
        between.append((reps[i], reps[j], int(hashes[reps[i]] - hashes[reps[j]])))

print("Within-cluster (same meme, modified):")
for a, b, d in within[:10]:
    print(f"  {d:3d}  {a}  vs  {b}")

print("\nBetween-cluster (different memes):")
for a, b, d in between[:10]:
    print(f"  {d:3d}  {a}  vs  {b}")

if within:
    print(f"\nWithin  mean={sum(d for _,_,d in within)/len(within):.1f}  n={len(within)}")
if between:
    print(f"Between mean={sum(d for _,_,d in between)/len(between):.1f}  n={len(between)}")

## 4. Quick histogram

Two distributions should be visibly separated if the algorithm works.

In [ ]:
import matplotlib.pyplot as plt

within_d = [d for _, _, d in within]
between_d = [d for _, _, d in between]

plt.figure(figsize=(8, 4))
plt.hist(within_d, bins=range(0, 257, 8), alpha=0.6, label=f"within-cluster (n={len(within_d)})")
plt.hist(between_d, bins=range(0, 257, 8), alpha=0.6, label=f"between-cluster (n={len(between_d)})")
plt.axvline(128, linestyle="--", color="grey", label="random baseline (128)")
plt.axvline(90, linestyle=":", color="red", label="typical threshold (~90)")
plt.xlabel("Hamming distance (bits)")
plt.ylabel("count")
plt.legend()
plt.title("FINd hash distances: same meme vs different memes")
plt.show()